# Inferential Statistical Tests

Complementary statistical analyses for the bachelor's thesis on ABPM hemodynamic coupling.
Covers paired within-subject comparisons, group contrasts, hypothesis-driven correlations,
Friedman and Kruskal-Wallis tests, and effect-size forest plots.

In [ ]:
import sys
import os

# Ensure CWD is project root so Config relative paths resolve correctly
os.chdir(os.path.join(os.path.dirname(os.path.abspath("__file__")), ".."))
sys.path.insert(0, ".")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats
from statsmodels.stats.multitest import fdrcorrection

from src.config import Config, Columns
from analysis.thesis_stats import (
    rank_biserial, bootstrap_ci, bootstrap_corr_ci,
    format_apa_wilcoxon, format_apa_mannwhitney, format_apa_spearman,
    format_apa_friedman, format_apa_kruskal, format_median_iqr,
    derive_dipper_category, export_table
)

config = Config()
THESIS_DIR = Path("results/thesis")
THESIS_DIR.mkdir(parents=True, exist_ok=True)
DPI = 300
sns.set_theme(style="whitegrid", font_scale=1.1)

In [ ]:
# Load data sources
sc = pd.read_csv(THESIS_DIR / "subject_context_medians.csv")
agg = pd.read_csv(THESIS_DIR / "aggregated_clean.csv")
res = pd.read_csv(config.get_results_path(config.SUBJECT_METRICS_OUTPUT))

print(f"Subject-context medians: {sc.shape}")
print(f"Aggregated clean: {agg.shape}")
print(f"Per-subject metrics: {res.shape}")

# Pivot subject_context_medians to wide format for paired comparisons
sc_wide = sc.pivot_table(
    index="participant_id", columns="label",
    values=["SBP_med", "DBP_med", "HR_med"],
)
sc_wide.columns = [f"{val}_{lab}" for val, lab in sc_wide.columns]
sc_wide = sc_wide.reset_index()
print(f"\nWide format: {sc_wide.shape}")
print(f"Columns: {list(sc_wide.columns)}")

## Section 1: Paired Within-Subject Comparisons (Baseline vs Conditions)

In [ ]:
# Wilcoxon signed-rank tests for raw hemodynamics
conditions = [
    Columns.LABEL_SLEEP,
    Columns.LABEL_COGNITIVE_TASK,
    Columns.LABEL_PHYSICAL_TASK,
    Columns.LABEL_AIR_ALERT,
]
outcomes = ["SBP_med", "DBP_med", "HR_med"]
outcome_labels = ["SBP", "DBP", "HR"]

paired_results = []

for condition in conditions:
    for outcome, out_label in zip(outcomes, outcome_labels):
        baseline_col = f"{outcome}_{Columns.LABEL_BASELINE}"
        condition_col = f"{outcome}_{condition}"

        if baseline_col not in sc_wide.columns or condition_col not in sc_wide.columns:
            continue

        # Only subjects with data in both conditions
        mask = sc_wide[baseline_col].notna() & sc_wide[condition_col].notna()
        baseline_vals = sc_wide.loc[mask, baseline_col].values
        condition_vals = sc_wide.loc[mask, condition_col].values
        n_paired = int(mask.sum())

        if n_paired < 5:
            continue

        diff = condition_vals - baseline_vals

        # Rank-biserial effect size (paired)
        r_rb, w_stat, p_val = rank_biserial(diff, paired=True)

        # Bootstrap CI for the median difference (fast), then map to r_rb CI
        # by resampling differences and computing r_rb via the simple formula
        rng = np.random.default_rng(42)
        n_boot = 2000
        boot_rbs = np.empty(n_boot)
        for i in range(n_boot):
            boot_diff = rng.choice(diff, size=len(diff), replace=True)
            boot_diff_nz = boot_diff[boot_diff != 0]
            if len(boot_diff_nz) < 2:
                boot_rbs[i] = 0.0
                continue
            n_nz = len(boot_diff_nz)
            ranks = stats.rankdata(np.abs(boot_diff_nz))
            w_plus = np.sum(ranks[boot_diff_nz > 0])
            w_total = n_nz * (n_nz + 1) / 2
            # simple rank-biserial: r = 4*W+/n(n+1) - 1
            boot_rbs[i] = 4 * w_plus / (n_nz * (n_nz + 1)) - 1

        ci_low = float(np.percentile(boot_rbs, 2.5))
        ci_high = float(np.percentile(boot_rbs, 97.5))

        direction = "increase" if np.median(diff) > 0 else "decrease"

        paired_results.append({
            "Comparison": f"{condition} vs Baseline",
            "Outcome": out_label,
            "N": n_paired,
            "W": w_stat,
            "p": p_val,
            "r_rb": r_rb,
            "CI_low": ci_low,
            "CI_high": ci_high,
            "Direction": direction,
        })

paired_df = pd.DataFrame(paired_results)
print(f"Paired comparisons computed: {len(paired_df)}")
paired_df

In [ ]:
# FDR correction for the family of paired tests
if len(paired_df) > 0:
    rejected, q_values = fdrcorrection(paired_df["p"].values, alpha=0.05)
    paired_df["q"] = q_values
    paired_df["Significant (FDR)"] = rejected

print(f"Significant after FDR correction: {paired_df['Significant (FDR)'].sum()} / {len(paired_df)}")
paired_df.sort_values("p")

In [ ]:
# APA-formatted strings for paired comparisons
print("=" * 80)
print("APA-formatted Wilcoxon signed-rank results")
print("=" * 80)

paired_apa_strings = []
for _, row in paired_df.iterrows():
    apa = format_apa_wilcoxon(
        W=row["W"], p=row["p"], r_rb=row["r_rb"],
        ci_low=row["CI_low"], ci_high=row["CI_high"], n=row["N"],
    )
    full_str = f"{row['Comparison']} [{row['Outcome']}]: {apa}"
    paired_apa_strings.append(full_str)
    sig_marker = " *" if row.get("Significant (FDR)", False) else ""
    print(f"  {full_str}{sig_marker}")

paired_df["APA_String"] = [s.split(": ", 1)[1] for s in paired_apa_strings]

**Note on Air Alert subsample.** Only 11 of 28 participants had an air alert during monitoring (`air_alert_during_monitoring == 1`), so all paired comparisons involving the Air Alert context have effective *N* = 11. Post-hoc power analysis (notebook 05, Table S1) shows that only large effects (*d*<sub>z</sub> ≥ 0.94) are reliably detectable at this sample size. Interpret Air Alert effect sizes with caution and flag *N* = 11 in all thesis captions referencing these results.

## Section 2: Group Comparisons (Mann-Whitney U)

In [ ]:
# Subgroup contrasts on BP summaries
grouping_vars = [
    "is_female", "is_young", "monitoring_diagnosis_is_healthy",
    "sleep_less_than_7h", "working_under_stress",
]
group_outcomes = [
    "SBP_all_med", "DBP_all_med", "HR_all_med",
    "SBP_all_iqr", "DBP_all_iqr", "HR_all_iqr",
]

mw_results = []

for grp_var in grouping_vars:
    if grp_var not in agg.columns:
        print(f"Warning: {grp_var} not found in aggregated data, skipping.")
        continue

    for outcome in group_outcomes:
        if outcome not in agg.columns:
            continue

        data = agg[[grp_var, outcome]].dropna()
        g0 = data.loc[data[grp_var] == 0, outcome].values
        g1 = data.loc[data[grp_var] == 1, outcome].values

        if len(g0) < 3 or len(g1) < 3:
            continue

        r_rb, u_stat, p_val = rank_biserial(g0, g1, paired=False)

        # Bootstrap CI for rank-biserial (independent): fast direct formula
        combined = np.concatenate([g0, g1])
        labels_arr = np.array([0] * len(g0) + [1] * len(g1))
        rng = np.random.default_rng(42)
        n_boot = 2000
        boot_rbs = np.empty(n_boot)
        n_total = len(combined)
        n1, n2 = len(g0), len(g1)

        for i in range(n_boot):
            boot_idx = rng.choice(n_total, size=n_total, replace=True)
            a_boot = combined[boot_idx[labels_arr[boot_idx] == 0]]
            b_boot = combined[boot_idx[labels_arr[boot_idx] == 1]]
            if len(a_boot) < 2 or len(b_boot) < 2:
                boot_rbs[i] = 0.0
            else:
                u_b = stats.mannwhitneyu(a_boot, b_boot, alternative="two-sided").statistic
                boot_rbs[i] = 1 - (2 * u_b) / (len(a_boot) * len(b_boot))

        ci_low = float(np.percentile(boot_rbs, 2.5))
        ci_high = float(np.percentile(boot_rbs, 97.5))

        mw_results.append({
            "Group": grp_var,
            "Outcome": outcome,
            "N0": len(g0),
            "N1": len(g1),
            "U": u_stat,
            "p": p_val,
            "r_rb": r_rb,
            "CI_low": ci_low,
            "CI_high": ci_high,
        })

mw_df = pd.DataFrame(mw_results)
print(f"Mann-Whitney tests computed: {len(mw_df)}")

In [ ]:
# FDR correction for Mann-Whitney family
if len(mw_df) > 0:
    rejected, q_values = fdrcorrection(mw_df["p"].values, alpha=0.05)
    mw_df["q"] = q_values
    mw_df["Significant (FDR)"] = rejected

    print(f"Significant after FDR: {mw_df['Significant (FDR)'].sum()} / {len(mw_df)}")
    print("\nResults with q < 0.1:")
    flagged = mw_df[mw_df["q"] < 0.1].sort_values("p")
    if len(flagged) > 0:
        print(flagged.to_string(index=False))
    else:
        print("  None")

    # Generate APA strings
    apa_strs = []
    for _, row in mw_df.iterrows():
        apa = format_apa_mannwhitney(
            U=row["U"], p=row["p"], r_rb=row["r_rb"],
            ci_low=row["CI_low"], ci_high=row["CI_high"],
            n1=row["N0"], n2=row["N1"],
        )
        apa_strs.append(apa)
    mw_df["APA_String"] = apa_strs

mw_df.sort_values("p")

## Section 3: Spearman Correlation Analysis

In [ ]:
# Hypothesis-driven Spearman correlations with bootstrap CIs
predictor_cols = [
    "age", "bmi", "bp_load_%", "sbp_dip_%", "dbp_dip_%",
    "sleep_duration_h", "DBP_all_med", "DBP_all_iqr",
]
corr_outcome_cols = [
    "DBP_Cognitive Task_DeltaBias", "DBP_Cognitive Task_Anomaly",
    "DBP_Physical Task_DeltaBias", "DBP_Physical Task_Anomaly",
    "DBP_Air Alert_DeltaBias", "DBP_Air Alert_Anomaly",
]

# Merge aggregated data with pipeline results
merged = agg.merge(res, on="participant_id", how="inner")
print(f"Merged dataset: {len(merged)} subjects")

corr_results = []

for pred in predictor_cols:
    for out in corr_outcome_cols:
        pair = merged[[pred, out]].dropna()
        n_pair = len(pair)
        if n_pair < 5:
            continue

        rho, ci_low, ci_high, p_val = bootstrap_corr_ci(
            pair[pred].values, pair[out].values,
            method="spearman", n_boot=2000, seed=42,
        )

        corr_results.append({
            "Predictor": pred,
            "Outcome": out,
            "N": n_pair,
            "rho": rho,
            "CI_low": ci_low,
            "CI_high": ci_high,
            "p": p_val,
        })

corr_df = pd.DataFrame(corr_results)
corr_df["abs_rho"] = corr_df["rho"].abs()
print(f"Correlation pairs computed: {len(corr_df)}")

**Note.** Correlations involving `DBP_Air Alert_DeltaBias` and `DBP_Air Alert_Anomaly` are computed on the *N* = 11 subsample with air-alert data. Reduced sample size inflates confidence-interval width and limits statistical power; treat these correlations as exploratory.

In [ ]:
# FDR correction within the correlation family
if len(corr_df) > 0:
    rejected, q_values = fdrcorrection(corr_df["p"].values, alpha=0.05)
    corr_df["q"] = q_values
    corr_df["Significant (FDR)"] = rejected

    # APA strings
    apa_strs = []
    for _, row in corr_df.iterrows():
        apa = format_apa_spearman(
            rs=row["rho"], p=row["p"],
            ci_low=row["CI_low"], ci_high=row["CI_high"], n=row["N"],
        )
        apa_strs.append(apa)
    corr_df["APA_String"] = apa_strs

    print(f"Significant after FDR: {corr_df['Significant (FDR)'].sum()} / {len(corr_df)}")
    print("\nTop correlations by |rho|:")

corr_df.sort_values("abs_rho", ascending=False).head(15)

## Section 4: Friedman Test (Multi-Condition)

In [ ]:
# Friedman test for subjects with data in Baseline, Cognitive Task, Physical Task
friedman_conditions = [
    Columns.LABEL_BASELINE,
    Columns.LABEL_COGNITIVE_TASK,
    Columns.LABEL_PHYSICAL_TASK,
]

friedman_results = []

for outcome, out_label in zip(outcomes, outcome_labels):
    # Get columns for each condition
    cols = [f"{outcome}_{cond}" for cond in friedman_conditions]
    missing_cols = [c for c in cols if c not in sc_wide.columns]
    if missing_cols:
        print(f"Skipping {out_label}: missing columns {missing_cols}")
        continue

    # Subjects with data in all three conditions
    mask = sc_wide[cols].notna().all(axis=1)
    n_subj = int(mask.sum())

    if n_subj < 5:
        print(f"Skipping {out_label}: only {n_subj} complete subjects")
        continue

    data_arrays = [sc_wide.loc[mask, c].values for c in cols]
    k = len(friedman_conditions)

    chi2, p_val = stats.friedmanchisquare(*data_arrays)
    chi2 = float(chi2)
    p_val = float(p_val)

    # Kendall's W effect size
    w_kendall = chi2 / (n_subj * (k - 1))

    apa = format_apa_friedman(
        chi2=chi2, p=p_val, W=w_kendall, df=k - 1, n=n_subj,
    )

    friedman_results.append({
        "Outcome": out_label,
        "N": n_subj,
        "k": k,
        "chi2": chi2,
        "p": p_val,
        "W_Kendall": w_kendall,
        "APA_String": apa,
    })

    print(f"{out_label}: {apa}")

    # Post-hoc pairwise Wilcoxon with BH correction if significant
    if p_val < 0.05:
        print(f"  -> Significant. Post-hoc pairwise Wilcoxon:")
        posthoc_ps = []
        posthoc_pairs = []
        for i in range(k):
            for j in range(i + 1, k):
                diff_ij = data_arrays[j] - data_arrays[i]
                diff_nz = diff_ij[diff_ij != 0]
                if len(diff_nz) < 2:
                    continue
                w_res = stats.wilcoxon(diff_nz)
                posthoc_ps.append(float(w_res.pvalue))
                posthoc_pairs.append(
                    f"{friedman_conditions[i]} vs {friedman_conditions[j]}"
                )

        if posthoc_ps:
            ph_rejected, ph_q = fdrcorrection(posthoc_ps, alpha=0.05)
            for pair, p_ph, q_ph, sig in zip(
                posthoc_pairs, posthoc_ps, ph_q, ph_rejected
            ):
                sig_str = "*" if sig else ""
                print(f"    {pair}: p = {p_ph:.4f}, q = {q_ph:.4f} {sig_str}")

friedman_df = pd.DataFrame(friedman_results)
print(f"\nFriedman tests computed: {len(friedman_df)}")

## Section 5: Kruskal-Wallis (Dipping Categories)

In [ ]:
# Kruskal-Wallis for dipping categories vs pipeline response metrics
merged["sbp_dipper_cat"] = derive_dipper_category(merged["sbp_dip_%"]).astype(str)

kw_outcomes = [
    "DBP_Cognitive Task_DeltaBias",
    "DBP_Cognitive Task_Anomaly",
]

kw_results = []

for outcome in kw_outcomes:
    data = merged[["sbp_dipper_cat", outcome]].dropna()

    # Count per category, keep only those with >= 3 subjects
    cat_counts = data["sbp_dipper_cat"].value_counts()
    valid_cats = cat_counts[cat_counts >= 3].index.tolist()

    if len(valid_cats) < 2:
        # Merge small categories: combine Reverse + Non-dipper, Normal + Extreme
        cat_map = {
            "Reverse dipper": "Non/Reverse dipper",
            "Non-dipper": "Non/Reverse dipper",
            "Normal dipper": "Normal/Extreme dipper",
            "Extreme dipper": "Normal/Extreme dipper",
        }
        data = data.copy()
        data["sbp_dipper_cat"] = data["sbp_dipper_cat"].map(cat_map)
        cat_counts = data["sbp_dipper_cat"].value_counts()
        valid_cats = cat_counts[cat_counts >= 3].index.tolist()
        print(f"{outcome}: Merged categories -> {dict(cat_counts)}")

    if len(valid_cats) < 2:
        print(f"Skipping {outcome}: insufficient groups after merging")
        continue

    data = data[data["sbp_dipper_cat"].isin(valid_cats)]
    groups = [g[outcome].values for name, g in data.groupby("sbp_dipper_cat", observed=True)]
    group_names = [name for name, _ in data.groupby("sbp_dipper_cat", observed=True)]

    # Filter out any empty groups
    non_empty = [(name, g) for name, g in zip(group_names, groups) if len(g) >= 2]
    if len(non_empty) < 2:
        print(f"Skipping {outcome}: fewer than 2 non-empty groups")
        continue
    group_names, groups = zip(*non_empty)
    group_names = list(group_names)
    groups = list(groups)

    n_total = sum(len(g) for g in groups)
    k = len(groups)

    h_stat, p_val = stats.kruskal(*groups)
    h_stat = float(h_stat)
    p_val = float(p_val)

    # Epsilon-squared
    epsilon_sq = h_stat / ((n_total ** 2 - 1) / (n_total + 1)) if n_total > 1 else 0.0

    apa = format_apa_kruskal(
        H=h_stat, p=p_val, epsilon_sq=epsilon_sq, df=k - 1, n=n_total,
    )

    kw_results.append({
        "Outcome": outcome,
        "N": n_total,
        "k": k,
        "H": h_stat,
        "p": p_val,
        "epsilon_sq": epsilon_sq,
        "Categories": ", ".join(group_names),
        "APA_String": apa,
    })

    print(f"{outcome}: {apa}")
    for name, grp in zip(group_names, groups):
        print(f"  {name} (n={len(grp)}): {format_median_iqr(grp)}")

    # Post-hoc pairwise Mann-Whitney with BH correction if significant
    if p_val < 0.05:
        print(f"  -> Significant. Post-hoc pairwise Mann-Whitney:")
        posthoc_ps = []
        posthoc_pairs = []
        for i in range(k):
            for j in range(i + 1, k):
                u_res = stats.mannwhitneyu(
                    groups[i], groups[j], alternative="two-sided",
                )
                posthoc_ps.append(float(u_res.pvalue))
                posthoc_pairs.append(f"{group_names[i]} vs {group_names[j]}")

        if posthoc_ps:
            ph_rejected, ph_q = fdrcorrection(posthoc_ps, alpha=0.05)
            for pair, p_ph, q_ph, sig in zip(
                posthoc_pairs, posthoc_ps, ph_q, ph_rejected
            ):
                sig_str = "*" if sig else ""
                print(f"    {pair}: p = {p_ph:.4f}, q = {q_ph:.4f} {sig_str}")

kw_df = pd.DataFrame(kw_results)
print(f"\nKruskal-Wallis tests computed: {len(kw_df)}")

**Figure 7 caption.** Forest plot of rank-biserial effect sizes (*r*<sub>rb</sub>) with 95% bootstrap CIs for within-subject paired comparisons (condition vs Baseline). †Air Alert comparisons are based on *N* = 11 (vs *N* = 28 for other conditions); wider CIs reflect reduced precision.

## Section 6: Effect Size Forest Plot

**Table 4 footnote for thesis.** Rows where Comparison contains "Air Alert" have effective *N* = 11. All other paired comparisons use the full *N* = 28 cohort. Report this in the table caption: "†*N* = 11 for Air Alert comparisons."

In [ ]:
# Forest plot of rank-biserial effect sizes from Section 1
if len(paired_df) > 0:
    plot_df = paired_df.sort_values("r_rb").reset_index(drop=True)
    plot_df["label"] = plot_df["Comparison"] + " [" + plot_df["Outcome"] + "]"

    outcome_colors = {"SBP": "#e74c3c", "DBP": "#3498db", "HR": "#2ecc71"}

    fig, ax = plt.subplots(figsize=(10, max(4, len(plot_df) * 0.45)))

    y_pos = range(len(plot_df))

    for i, (_, row) in enumerate(plot_df.iterrows()):
        color = outcome_colors[row["Outcome"]]
        # Ensure xerr is non-negative
        err_low = max(0, row["r_rb"] - row["CI_low"])
        err_high = max(0, row["CI_high"] - row["r_rb"])
        ax.errorbar(
            row["r_rb"], i,
            xerr=[[err_low], [err_high]],
            fmt="o", color=color, ecolor=color, elinewidth=2,
            capsize=4, markersize=7, markeredgecolor="white", markeredgewidth=0.5,
        )

    ax.axvline(0, color="black", linestyle="--", linewidth=1, alpha=0.7)
    ax.set_yticks(list(y_pos))
    ax.set_yticklabels(plot_df["label"], fontsize=9)
    ax.set_xlabel("Rank-biserial correlation ($r_{rb}$)")
    ax.set_title("Effect Sizes: Paired Within-Subject Comparisons")

    # Legend
    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], marker="o", color="w", markerfacecolor=c, markersize=8, label=k)
        for k, c in outcome_colors.items()
    ]
    ax.legend(handles=legend_elements, loc="lower right", fontsize=9)

    plt.tight_layout()
    fig.savefig(
        THESIS_DIR / "fig_07_effect_size_forest.png",
        dpi=DPI, bbox_inches="tight",
    )
    plt.show()
    plt.close(fig)
    print(f"Saved: {THESIS_DIR / 'fig_07_effect_size_forest.png'}")
else:
    print("No paired results to plot.")

## Section 7: Export Results

In [ ]:
# Combine all inferential results into a master table
master_rows = []

# Paired comparisons (Wilcoxon)
for _, row in paired_df.iterrows():
    master_rows.append({
        "Family": "Paired (Wilcoxon)",
        "Comparison": row["Comparison"],
        "Outcome": row["Outcome"],
        "Test": "Wilcoxon signed-rank",
        "N": row["N"],
        "Statistic": f"W = {row['W']:.1f}",
        "Effect_Size": f"r_rb = {row['r_rb']:.2f}",
        "CI_95": f"[{row['CI_low']:.2f}, {row['CI_high']:.2f}]",
        "p": row["p"],
        "q": row.get("q", np.nan),
        "Direction": row["Direction"],
        "APA_String": row.get("APA_String", ""),
    })

# Mann-Whitney
for _, row in mw_df.iterrows():
    master_rows.append({
        "Family": "Group (Mann-Whitney)",
        "Comparison": row["Group"],
        "Outcome": row["Outcome"],
        "Test": "Mann-Whitney U",
        "N": row["N0"] + row["N1"],
        "Statistic": f"U = {row['U']:.1f}",
        "Effect_Size": f"r_rb = {row['r_rb']:.2f}",
        "CI_95": f"[{row['CI_low']:.2f}, {row['CI_high']:.2f}]",
        "p": row["p"],
        "q": row.get("q", np.nan),
        "Direction": "",
        "APA_String": row.get("APA_String", ""),
    })

# Friedman
for _, row in friedman_df.iterrows():
    master_rows.append({
        "Family": "Multi-condition (Friedman)",
        "Comparison": "Baseline vs Cognitive vs Physical",
        "Outcome": row["Outcome"],
        "Test": "Friedman",
        "N": row["N"],
        "Statistic": f"chi2 = {row['chi2']:.2f}",
        "Effect_Size": f"W = {row['W_Kendall']:.2f}",
        "CI_95": "",
        "p": row["p"],
        "q": np.nan,
        "Direction": "",
        "APA_String": row.get("APA_String", ""),
    })

# Kruskal-Wallis
for _, row in kw_df.iterrows():
    master_rows.append({
        "Family": "Dipping (Kruskal-Wallis)",
        "Comparison": row["Categories"],
        "Outcome": row["Outcome"],
        "Test": "Kruskal-Wallis",
        "N": row["N"],
        "Statistic": f"H = {row['H']:.2f}",
        "Effect_Size": f"eps2 = {row['epsilon_sq']:.3f}",
        "CI_95": "",
        "p": row["p"],
        "q": np.nan,
        "Direction": "",
        "APA_String": row.get("APA_String", ""),
    })

master_df = pd.DataFrame(master_rows)
print(f"Master table: {master_df.shape}")

export_table(master_df, THESIS_DIR / "table_04_inferential_effect_sizes")
print(f"Saved: table_04_inferential_effect_sizes.csv / .tex")
master_df

In [ ]:
# Export correlation results separately
if len(corr_df) > 0:
    corr_export = corr_df.drop(columns=["abs_rho"]).sort_values("p")
    export_table(corr_export, THESIS_DIR / "table_S2_correlation_results")
    print(f"Saved: table_S2_correlation_results.csv / .tex")
    print(f"Rows: {len(corr_export)}")
    corr_export.head(10)
else:
    print("No correlation results to export.")